# W4C2: Word embeddings, and the arithmetic that made them famous

Run every cell from the top. **Everything already works.**

These vectors are GloVe, trained on Wikipedia and a decade of newswire. Nobody labelled them and nobody chose what the 50 numbers mean. Each part ends with a **TRY IT**: one question, one empty cell.

Today you will:

1. Open a **real** embedding: 20,000 words, 50 numbers each, learned from text.
2. Find a word's neighbours with **cosine similarity**.
3. Do the arithmetic: **king - man + woman**.
4. Do it again with `doctor`, and look at what comes out.

Nothing to submit. Answers are in the last cell.

In [ ]:
# Setup. Run this cell first.
import numpy as np
import pandas as pd

data = np.load("data/glove-50d-20k.npz")
WORDS = [str(w) for w in data["words"]]      # most frequent first
VECTORS = data["vectors"]                    # (20000, 50) float32
INDEX = {word: i for i, word in enumerate(WORDS)}

print("VECTORS.shape:", VECTORS.shape)
print("the ten most common words:", WORDS[:10])

## Part 1. A word is 50 numbers


In a **sparse** representation a word is one slot in a long vocabulary, and a
document is a row that is almost entirely zeros. Here a word **is** the row: 50
numbers, every one of them non-zero, none of them chosen by a person.

Nobody decided what the 50 numbers mean. They were fitted so that words appearing
in similar contexts ended up close together, which is the distributional
hypothesis turned into arithmetic.


<img src="images/sparse-vs-dense.png" width="640">

In [ ]:
def vector(word):
    """The 50 numbers for one word."""
    return VECTORS[INDEX[word]]


king = vector("king")

print("vector('king') is", king.shape, "of", king.dtype)
print()
print(np.round(king[:6], 3), "... 44 more")
print()
print("No slot means anything on its own. Slot 3 is not 'royalty'.")
print("The information is in how the whole row compares with another whole row.")

In [ ]:
# ================== TRY IT 1 ==================
# In a sparse representation over a 20,000-word vocabulary, how many
# numbers would `king` have, and how many would be non-zero?
# How many are non-zero here?
# ==============================================


## Part 2. Comparing two words


Cosine similarity measures the angle between two rows. It does not care how many
numbers the rows have, or whether they were counted or learned.


<img src="images/cosine-similarity.png" width="640">

In [ ]:
def cosine(first, second):
    """The cosine of the angle between two vectors."""
    lengths = np.linalg.norm(first) * np.linalg.norm(second)
    if lengths == 0:
        return 0.0
    return np.dot(first, second) / lengths


for a, b in [("king", "queen"), ("cat", "dog"), ("king", "cat")]:
    print(f"  {a:5s} {b:6s} {cosine(vector(a), vector(b)):.3f}")
print()
print("Sparse vectors sharing no word at all score exactly 0.000.")
print("Here nothing is ever 0: every word has some angle to every other word.")

In [ ]:
# Normalise every row once, and one matrix multiply scores all 20,000 words.
NORMALISED = VECTORS / np.linalg.norm(VECTORS, axis=1, keepdims=True)


def nearest(word, how_many=5):
    """The how_many words closest to this one."""
    scores = NORMALISED @ (vector(word) / np.linalg.norm(vector(word)))
    order = np.argsort(-scores)
    found = [(WORDS[i], round(float(scores[i]), 3))
             for i in order if WORDS[i] != word]
    return found[:how_many]


for word in ["king", "cat", "paris", "october"]:
    print(f"  {word:8s}", nearest(word))


`october` is worth a second look. Its five nearest words are the other months,
all above 0.99, because a month is used in almost exactly the same sentences as
any other month. Nobody told it what a month is.

And `doctor`'s nearest neighbour is `nurse`. Hold on to that.


In [ ]:
# ================== TRY IT 2 ==================
# Pick a word you expect to have obvious neighbours and one you expect
# to have strange ones. Were you right?
# ==============================================


## Part 3. king - man + woman


Here is the result that made word embeddings famous in 2013.

If the direction from `man` to `woman` is the same as the direction from `king`
to `queen`, then you can move between them by adding and subtracting whole
vectors. It is a strong claim, and it is testable in three lines.


In [ ]:
# ================== YOUR TURN 1 ==================
# Finish `analogy`. Take the step from `a` to `b`, apply it starting
# at `c`, and return the nearest words to where you land.
#
# The step from a to b is `vector(b) - vector(a)`. Applying it at c means
# adding `vector(c)`. One line.
#
# Hint: target = vector(b) - vector(a) + vector(c)
#
# Expected: king - man + woman -> queen at 0.861, well clear of daughter at
#           0.768. Until you change the line it returns the neighbours of
#           `woman` alone: girl 0.907, mother 0.876, her 0.861.
# =================================================
def analogy(a, b, c, how_many=3):
    """b is to a as ??? is to c."""
    target = vector(c)                      # <-- change this line

    scores = NORMALISED @ (target / np.linalg.norm(target))
    order = np.argsort(-scores)
    skip = {a, b, c}
    found = [(WORDS[i], round(float(scores[i]), 3))
             for i in order if WORDS[i] not in skip]
    return found[:how_many]


print("king - man + woman ->", analogy("man", "king", "woman"))

In [ ]:
# Needs your line above. The same three lines, on two other relationships.
print("paris - france + italy   ->", analogy("france", "paris", "italy"))
print("walking - walk + swim    ->", analogy("walk", "walking", "swim"))

In [ ]:
# Also needs your line. And these are the ones nobody puts on a slide.
print("brother - man + woman     ->", analogy("man", "brother", "woman"))
print("washington - usa + france ->", analogy("usa", "washington", "france"))
print("bigger - big + small      ->", analogy("big", "bigger", "small"))


`sister` is not in the first list. `paris` is not in the second. `smaller` is
third in the third, behind two words that are not even the right shape of answer.

The famous example is real, and it is also **cherry-picked**. Analogy arithmetic
works often enough to be interesting and fails often enough that you should never
report the one that worked without saying how many you tried.


In [ ]:
# ================== TRY IT 3 ==================
# Try three analogies of your own. How many gave you the answer you
# expected in first place?
# ==============================================


## Part 4. The same arithmetic, on a job


Every line below uses the function you just wrote. Nothing changes except the
words going in.


In [ ]:
print("doctor - man + woman   ->", analogy("man", "doctor", "woman"))
print("engineer - man + woman ->", analogy("man", "engineer", "woman"))
print("genius - man + woman   ->", analogy("man", "genius", "woman"))


Nobody wrote that down. It was fitted from how people actually write, and it will
be fitted again from any corpus of human text you can find.

One vector holds the direction itself: `she - he`. Score a word against it and
you get how far that word leans, which is a measurement rather than an anecdote.


In [ ]:
# ================== YOUR TURN 2 ==================
# Measure it instead of eyeballing it.
#
# `gender_direction` is `she - he`. Score each job against it with `cosine`
# and sort, most positive first. Fill in the marked line.
#
# Hint: score = cosine(vector(job), gender_direction)
#
# Expected: nurse +0.381 and dancer +0.357 at the top, captain -0.316 and
#           boss -0.246 at the bottom, and doctor +0.124 against engineer
#           -0.125. Until you change the line every score is 0.000.
# =================================================
gender_direction = vector("she") - vector("he")

JOBS = ["nurse", "dancer", "teacher", "librarian", "doctor", "surgeon",
        "lawyer", "scientist", "programmer", "engineer", "banker",
        "secretary", "architect", "boss", "captain"]

leaning = []
for job in JOBS:
    score = 0.0                             # <-- change this line
    leaning.append({"job": job, "she - he": round(float(score), 3)})

table = pd.DataFrame(leaning).sort_values("she - he", ascending=False)
print(table.to_string(index=False))


Read the bottom of that table before the top. `secretary` lands at **-0.216**,
the male end, which is the opposite of the stereotype.

These vectors were trained on Wikipedia and newswire, where "secretary" is mostly
the Secretary of State or of Defense. The model did not learn about the world. It
learned about **a pile of text somebody chose**, and a different pile would have
given a different answer.

That is the question you have after the break: if the corpus decides, what would
you change, and what would it cost you?


## Answers

Try each task before reading.

In [ ]:
# TRY IT 1
#   Sparse: one slot per vocabulary word, so 20,000 numbers, of which a handful
#   are non-zero. Dense: 50 numbers, and every one of them is non-zero.
#   np.count_nonzero(vector("king"))  ->  50
#   Sparse and interpretable became dense and uninterpretable, and that trade is
#   the whole reason embeddings work.

# TRY IT 2
#   Anything goes, but two patterns are worth naming out loud.
#   nearest("october")  ->  the other months, all above 0.99, because months
#     appear in near-identical sentences.
#   nearest("apple")    ->  blackberry, chips, iphone, microsoft, pc, ipod.
#     Not one fruit. The company sense has swallowed the word whole, because
#     that is how "apple" is used in news text. Same for python: owl, mouse,
#     monkey, monty, and no programming language anywhere. One word gets one
#     vector, so the senses average and the rarer one loses. That is the
#     static-embedding limitation, and contextual embeddings are what fix it.

# TRY IT 3
#   Expect one or two of three. Relationships that are strongly and consistently
#   written down (capitals, verb tense, comparatives) tend to work; family
#   relations and anything needing world knowledge tend not to.

# YOUR TURN 1
#   target = vector(b) - vector(a) + vector(c)
#
#   king - man + woman        ->  queen 0.861, daughter 0.768, prince 0.764
#   paris - france + italy    ->  rome 0.838, milan 0.772, turin 0.760
#   walking - walk + swim     ->  swimming 0.807, swimmers 0.756, surfing 0.742
#
#   and the failures:
#   brother - man + woman     ->  daughter 0.937, wife 0.913, mother 0.899
#                                 (sister is not in the top three)
#   washington - usa + france ->  chirac 0.734, diplomatic 0.722, brussels 0.715
#                                 (paris is not there at all)
#   bigger - big + small      ->  larger 0.904, large 0.871, smaller 0.866

# YOUR TURN 2
#   score = cosine(vector(job), gender_direction)
#
#     nurse       +0.381        scientist   -0.039
#     dancer      +0.357        lawyer      -0.061
#     doctor      +0.124        banker      -0.092
#     librarian   +0.109        engineer    -0.125
#     teacher     +0.104        secretary   -0.216
#     surgeon     +0.053        architect   -0.228
#     programmer  -0.033        boss        -0.246
#                               captain     -0.316
#
#   The top is the stereotype you expected. The bottom is more interesting:
#   secretary sits at the male end because this corpus is news, and in news a
#   secretary is a cabinet minister. Change the corpus, change the bias. That is
#   the whole argument for the exercise after the break.

# The three things worth carrying out of today:
#   1. A word is a dense row of learned numbers. No slot means anything alone;
#      only the comparison between two whole rows does.
#   2. The arithmetic really works, and it really fails, and you only ever hear
#      about the first kind.
#   3. The bias is not a bug in the algorithm. It is an accurate summary of the
#      text it was given, which is why the fix has to be about the text.